# Chennai PG Intelligence: Project Overview

> A plain English walkthrough of everything I did througout the EDA, that tells what I found, and where i'm headed into.
> No prior knowledge of data science or machine learning is needed to read this document.

---

## Table of Contents

1. [What Is This Project?](#1-what-is-this-project)
2. [The Dataset](#2-the-dataset)
3. [How We Approached the Problem](#3-how-we-approached-the-problem)
4. [Phase Breakdowns](#4-phase-breakdowns)
   - [Phase 1: First Look at the Raw Data](#phase-1-first-look-at-the-raw-data)
   - [Phase 2: Cleaning the Data](#phase-2-cleaning-the-data)
   - [Phase 3: Understanding Each Feature Individually](#phase-3-understanding-each-feature-individually)
   - [Phase 4: Which Features Actually Affect Price?](#phase-4-which-features-actually-affect-price)
   - [Phase 5: How Do Features Behave Together?](#phase-5-how-do-features-behave-together)
5. [Key Takeaways: What the Data Told Us](#5-key-takeaways-what-the-data-told-us)
6. [The Final Dataset](#6-the-final-dataset)
7. [What Is Next](#7-what-is-next)
8. [Glossary](#8-glossary)

---

## 1. What Is This Project?

A **PG** (short for *Paying Guest*) is a type of rented accommodation very common across Indian cities. You rent a room (sometimes shared with others) and typically get basic amenities like Wi-Fi, food, and laundry included in the monthly price.

Finding a fair priced PG in Chennai is genuinely hard. The same locality can have PGs charging ₹5,000 and others charging ₹15,000, and it's not always obvious why. Tenants end up overpaying because they have no reference point, and owners sometimes misprice because they have no systematic way to compare.

**This project builds a machine learning model that predicts the monthly rent of a Chennai PG** given information about it: its location, the type of room, who it is meant for, and what amenities it offers.

Once trained, such a model can:
- Help a tenant quickly check if a listed price is fair for what's on offer
- Help a PG owner price a room competitively based on comparable listings
- Surface the specific features that drive price up or down in each locality

---

## 2. The Dataset

The data was collected from **NoBroker**, one of India's largest property listing platforms, using their public listing API. Each row in the dataset represents a single **room type within a PG property**. For example, if a PG offers both single sharing and double sharing rooms, it appears as two separate rows.

### At a glance

| | |
|---|---|
| Source | NoBroker public listing API |
| City | Chennai, Tamil Nadu |
| Raw listings | 1,783 rows |
| Localities covered | 36 |
| Price range (rent) | ₹1,500 to ₹65,000 per month |
| Typical rent | ₹7,000 per month (median) |

### What information each listing contains

Every listing comes with details across four broad areas:

**Location**
Where the PG is: the locality (like Velachery or Tharamani), the street address, and GPS coordinates (latitude and longitude). NoBroker also provides a *transit score* (how well connected the area is to public transport) and a *lifestyle score* (how many cafes, gyms, restaurants etc. are nearby).

**Room details**
The type of room, whether it's single occupancy (you alone), double sharing, triple sharing, or four person sharing. Whether it is meant for males, females, or anyone.

**Amenities**
Whether the PG provides things like Wi-Fi, food (breakfast/lunch/dinner), power backup, laundry, a refrigerator, air conditioning in the room, a geyser, a TV, parking for a bike or car, and more.

**Pricing**
The monthly rent and the one time security deposit.

---

## 3. How We Approached the Problem

Before writing a single line of a predictive model, we spent time deeply understanding the data itself. This process is called **Exploratory Data Analysis** (EDA). Think of it as reading the entire rulebook before playing the game.

The goal of EDA is to answer questions like:
- Is the data complete, or are there gaps?
- Are there errors or impossible values hiding in the numbers?
- Which features genuinely influence the rent, and which ones don't?
- Are there surprising patterns or relationships we didn't expect?

We split this into **5 phases**, each building on the last:

| Phase | What it covered |
|---|---|
| Phase 1 | First inspection: shape, gaps, errors, duplicates |
| Phase 2 | Cleaning: fixing and removing everything Phase 1 flagged |
| Phase 3 | Univariate analysis: studying each feature one at a time |
| Phase 4 | Bivariate analysis: comparing each feature against rent |
| Phase 5 | Multivariate analysis: checking how features behave together |

---

## 4. Phase Breakdowns

---

### Phase 1: First Look at the Raw Data

**The goal:** open the raw dataset for the first time and catalogue everything that needs attention before we touch the data.

#### What we found

**Missing values: three different kinds of gaps**

Not all missing values mean the same thing, and treating them identically would be a mistake. We identified three distinct categories:

- *Transit and lifestyle scores were missing for about half the listings.* NoBroker simply hadn't computed scores for those localities. These aren't accidental gaps. NoBroker just doesn't cover every area. We kept the columns but made a note to handle this carefully.

- *Amenity columns (like Wi-Fi, laundry, power backup) were missing for about 45% of listings.* We traced this back to the raw NoBroker API data. When a PG owner doesn't fill in their amenities section, the API returns nothing, not a "No", just silence. So a missing value here means "the owner didn't say", not "this amenity doesn't exist." For the purposes of this model, we decided to treat silence as "not available", a conservative but defensible assumption.

- *About 1% of listings were missing values for rent, deposit, or room type.* These rows are too incomplete to be useful and too few in number to matter if we drop them. They were marked for removal.

**One column was almost entirely empty**
The `gate_closing_time` column (the time the PG closes its gate at night) was missing for over 80% of listings. Even if it were complete, it doesn't influence the monthly rent. It was marked for removal.

**Duplicate listings**
Some properties appeared more than once in the data, likely because NoBroker listed the same PG under multiple localities (for example, once under *Tambaram* and once under *Perungalathur*). These exact duplicates were identified and flagged.

**Impossible values**
The transit score had a minimum value of **negative 10**, which is not a real score. NoBroker uses negative 10 as a code meaning "no data available for this location." Treating negative 10 as an actual score would corrupt any analysis. It was flagged for replacement with a proper missing value marker.

The rent column had some entries of ₹0 or very close to zero. No PG in Chennai rents for zero. These are clearly data entry errors. Rows with rent below ₹1,000 were flagged for removal.

**Columns that carry no useful information for prediction**
The unique ID assigned to each listing, the free text listing title, and the full street address all carry no signal for predicting rent (the locality column already captures location). These were marked for removal.

#### Decisions made

| Item | Decision |
|---|---|
| id, title, address | Drop (no predictive value) |
| gate_closing_time | Drop (80%+ missing, low signal) |
| total_bathrooms | Drop (extreme outliers, redundant with other bathroom columns) |
| Transit and lifestyle scores | Keep, but impute missing values using the median for the same locality |
| Amenity columns | Keep, treat missing as "not available" |
| Rows with rent ≤ ₹1,000 or no rent | Drop |
| Duplicate listings | Remove, keep the first occurrence |

---

### Phase 2: Cleaning the Data

**The goal:** execute every decision from Phase 1 and produce a clean dataset for all further analysis.

This phase is less about discovery and more about careful execution. The key actions taken:

**Removing clutter**
Five identifier/free text columns were dropped (id, title, address, gate_closing_time, total_bathrooms). Five more columns were also dropped after closer inspection: `warden`, `cooking_allowed`, `guardian_required`, `nonveg_allowed`, and `smoking_allowed` all had very little variation across listings (almost every listing had the same answer), which means they would contribute almost nothing to a model.

**Removing bad rows**
Rows were dropped if they were missing the rent, deposit, room type, or bathroom information. Rows with rent below ₹1,000 were removed. All duplicate property and room combinations were removed. After this, the dataset went from 1,783 raw rows to a clean 1,471 rows.

**Fixing the amenity columns**
All 14 amenity columns were storing data in a mixed format that Python couldn't read reliably as True/False. They were standardised: any missing value was filled with `False`, and the column type was locked to a proper boolean (True/False) format.

**Fixing the transit score**
The negative 10 sentinel value was replaced with a proper "missing" marker. Then the gap was filled by computing the median score within the same locality. If Velachery's average transit score is 7.2, a Velachery listing with no score gets 7.2. If an entire locality had no scores at all, the gap was filled with the overall dataset median. A new column was also created to flag which rows originally had missing scores, so the model can learn whether being in an "unscored" locality is itself a meaningful signal.

The same two step strategy (indicator flag → locality median → global median fallback) was applied to the lifestyle score.

**Output**
A clean, validated dataset saved as `01_eda_Phase-2_processed.csv`, with 1,471 rows, 31 columns, zero missing values.

---

### Phase 3: Understanding Each Feature Individually

**The goal:** study one feature at a time: what does its distribution look like, is it skewed, are there outliers, does it need further attention?

#### Numeric features

**Rent (our target, the value we want to predict)**

| Metric | Value |
|---|---|
| Average | ₹7,779 |
| Median | ₹7,000 |
| Most common value | ₹6,500 |
| Lowest | ₹1,500 |
| Highest | ₹65,000 |

The rent distribution is skewed to the right, with most PGs clustering in the ₹5,000 to ₹9,000 range, but a small number of premium listings push the average up. This is completely realistic and not a data problem. The high end outliers are genuine premium PGs and were kept in.

**Deposit**

Deposits are even more skewed than rent, with the median at ₹4,000 but the maximum is ₹2,00,000. A deposit eight times the rent on a basic room turned out to be a data error (caught and removed in Phase 5). Otherwise, the variation is genuine.

**Transit and lifestyle scores**

Both columns turned out to have a two humped distribution, meaning instead of one smooth bell curve there are two separate peaks. This suggests Chennai's PG localities naturally split into two groups: well connected areas and less connected areas, rather than sitting on one smooth spectrum. This was noted for the modeling stage.

#### Categorical features

**Locality (36 areas)**
The top three localities, Tharamani (240 listings, 16.3%), OMR-Karapakkam (232 listings, 15.8%), and Velachery (192 listings, 13.1%), account for nearly half the entire dataset. About 24 of the 36 localities have fewer than 50 listings each. This long tail means the model will need to be careful not to overfit on the thin localities.

**Occupancy (room sharing type)**
Four types: THREE sharing (32.4%), FOUR sharing (27.3%), DOUBLE sharing (27.1%), SINGLE (13.2%). Single rooms are fewest because they're more expensive, so fewer people can afford them.

**Gender**
MALE (52.3%), FEMALE (44.9%), BOTH/Mixed (2.7%). Fairly balanced between male and female PGs.

**Available for**
88.85% of listings are open to "Both" (students and working professionals). Only 9.45% are exclusively for working professionals, and 1.7% for students only. This is very imbalanced but the feature wasn't dropped yet, and Phase 4 was needed to check if it actually shifts the price.

#### Boolean (Yes/No) features

A key finding here: the columns `breakfast`, `lunch`, and `dinner` were almost perfectly identical to the `food_included` column. When a PG includes food, it almost always includes all three meals together. Keeping all four columns would be redundant, so `breakfast`, `lunch`, and `dinner` were marked for removal, keeping only `food_included`.

---

### Phase 4: Which Features Actually Affect Price?

**The goal:** for every feature in the dataset, check whether it actually moves the rent needle and by how much.

#### Deposit vs. Rent

The strongest numeric relationship found. Higher deposits go with higher rents, with a correlation of 0.44 (on a scale of 0 to 1, where 1 means perfect relationship). This makes intuitive sense: a landlord charging more rent also charges a bigger deposit.

#### Occupancy vs. Rent

The single most powerful feature in the dataset. The pattern is exactly what you'd expect:

| Room type | Average rent |
|---|---|
| Single (just you) | Highest |
| Double sharing | Second highest |
| Three sharing | Third |
| Four sharing | Lowest |

The more people share a room, the less each person pays. This isn't just a small difference. Single rooms cost significantly more than four sharing rooms, and the gap varies dramatically by locality, as seen in Phase 5.

#### Locality vs. Rent

Locality has a real and substantial effect on price. PGs in better connected, more central areas charge more. The effect survives even when controlling for room type.

#### Food included vs. Rent

PGs that include food charge about ₹433 more on average than those that don't. A real but modest effect.

#### Parking vs. Rent

Car parking PGs appear to charge more than bike only ones, but the difference needed further investigation: was it genuinely the parking, or just because car parking PGs happened to have more single rooms (which are already the most expensive)? This was left open for Phase 5.

#### Gender vs. Rent

Mixed gender (BOTH) PGs appeared to charge more on average. The question was: was it the gender policy, or was it because most mixed PGs target working professionals who happen to pay more? Phase 5 investigated this.

#### Transit score, lifestyle score, latitude, longitude

All four showed weak or essentially flat relationships with rent. The city's transit infrastructure and lifestyle amenities don't neatly translate to higher rent in this dataset. These features were kept in but flagged as low priority.

#### A note on multicollinearity

Transit score and latitude are moderately correlated with each other (0.52), meaning they tend to move together. The model will need to account for this to avoid double counting.

---

### Phase 5: How Do Features Behave Together?

**The goal:** check how features behave *together*, not just in isolation. Settle the open questions from Phase 4.

#### Settling the parking question

In Phase 4, car parking PGs appeared more expensive, but were they pricier because of the parking itself, or just because they had more single rooms?

We ran a precise test: what would the average rent of Car parking listings be if they had *exactly the same mix of localities and room types* as the full dataset? The result:

| Parking type | Expected rent (adjusted) | Actual rent | Difference |
|---|---|---|---|
| Car | ₹8,099 | ₹9,144 | **+₹1,045** |
| Bike | ₹7,774 | ₹7,852 | +₹78 (noise) |
| Bike and Car | ₹7,899 | ₹7,458 | minus ₹442 |

Car parking carries a genuine premium of about ₹1,045 that cannot be explained by room type or location. Something about properties with car parking, possibly quality, space, or bundled amenities, makes them pricier.

Bike parking has zero independent effect. Bike and Car listings actually rent *below* what their location and room type would predict, a real but small effect.

#### Settling the gender question

The appearance that mixed gender (BOTH) PGs charge more was almost entirely driven by the fact that most BOTH-gender PGs specifically target working professionals, who pay more. Once you account for that, the gender effect largely disappears. The feature was kept but noted as a weak independent predictor.

#### Locality and Occupancy: the most important discovery of Phase 5

In Phase 4, we knew occupancy (room sharing type) affects rent. But we hadn't asked: *does the occupancy effect stay the same in every locality?*

It doesn't. Not even close.

| Locality | Rent gap: single vs. four sharing |
|---|---|
| Sholinganallur | ₹5,725 |
| Velachery | ₹5,623 |
| Tharamani | ₹4,955 |
| Karapakkam | ₹4,603 |
| Vadapalani | ₹2,546 |
| Tambaram | ₹1,893 |
| New Perungalathur | **₹75** |

In Sholinganallur, a single room costs ₹5,725 more than a four sharing room. In New Perungalathur, the same comparison is essentially ₹75, and room type barely matters to the price there. The interaction between locality and room type is one of the most important things a prediction model needs to capture.

#### Catching a data error

One listing kept appearing as an extreme outlier in every plot: a FOUR sharing room in Vadapalani with a ₹2,00,000 deposit, ₹25,000 rent, and zero amenities. A FOUR-sharing room (the cheapest type) with the highest deposit in the entire dataset and no amenities at all is internally contradictory. This was almost certainly an extra digit entered by mistake. The row was removed.

---

## 5. Key Takeaways: What the Data Told Us

These are the most important findings from five phases of analysis, in plain English:

- **Room type is the single biggest driver of price.** Single rooms consistently cost more than double, which costs more than triple, which costs more than four sharing, in every locality, without exception.

- **Location matters enormously, but not uniformly.** The rent difference between room types is huge in Sholinganallur (₹5,725 gap) and almost nonexistent in New Perungalathur (₹75 gap). A model needs to understand *both* the locality and the room type together, not just each one separately.

- **Deposit is the strongest numeric signal.** Higher rent and higher deposit go hand in hand (0.44 correlation). A large deposit on a listing is a reliable indicator that the rent is also high.

- **Car parking adds a genuine ~₹1,045 premium** that holds up even when accounting for location and room type. Bike parking has no independent effect.

- **Food inclusion adds a modest ~₹433 premium**, real but small.

- **Tharamani, OMR-Karapakkam, and Velachery dominate the dataset**, together accounting for nearly half of all listings. The model will be most reliable in these areas and less certain in thin localities.

- **Transit and lifestyle scores are weak predictors.** The scores exist for only about half the listings and don't correlate strongly with rent once location is already known.

- **Mixed gender PGs aren't independently more expensive.** The apparent premium comes from their working professional tenant mix, not the gender policy itself.

---

## 6. The Final Dataset

After five phases of analysis and all the cleaning decisions, here is where we stand heading into the next stage:

| | |
|---|---|
| Raw dataset | 1,783 rows |
| After removing duplicates and invalid rows | 1,471 rows |
| After removing the one confirmed data error | **1,470 rows** |
| Columns removed | 13 (identifiers, high missingness, redundant, near zero variance) |
| **Final size** | **1,470 rows × 28 columns** |

### What the 28 columns are

| Column | What it represents |
|---|---|
| `rent` | Monthly rent, the value we are predicting |
| `deposit` | One time security deposit |
| `locality` | Which area of Chennai the PG is in (36 possible values) |
| `occupancy` | How many people share the room (Single/Double/Three/Four) |
| `gender` | Whether the PG is for males, females, or anyone |
| `available_for` | Whether it's open to students, working professionals, or anyone |
| `parking` | What parking is available (none / bike / car / bike and car) |
| `latitude` & `longitude` | GPS coordinates of the PG |
| `transit_score` | How well connected the area is to public transport |
| `lifestyle_score` | How many nearby lifestyle amenities (cafes, gyms, etc.) |
| `transit_score_missing` | Flag: was the transit score originally absent? (Yes/No) |
| `lifestyle_score_missing` | Flag: was the lifestyle score originally absent? (Yes/No) |
| `food_included` | Does the PG include meals? |
| `attached_bathroom` | Is there an attached bathroom? |
| `wifi` | Is Wi-Fi provided? |
| `laundry` | Is laundry service available? |
| `power_backup` | Is there power backup? |
| `refrigerator` | Is a refrigerator available? |
| `common_tv` | Is there a common TV area? |
| `room_cleaning` | Is room cleaning included? |
| `room_ac` | Is the room air conditioned? |
| `room_cupboard` | Is there a cupboard in the room? |
| `room_tv` | Is there a TV in the room? |
| `room_geyser` | Is there a geyser in the room? |
| `room_bedding` | Is bedding provided? |
| `room_attached_bath` | Does the room have its own attached bathroom? |
| `mess` | Is a mess (common dining facility) available? |

---

## 7. What Is Next

The next step is **preprocessing**, which involves converting this clean, well understood dataset into a form that a machine learning model can actually read and learn from.

Concretely, this means:

- **Encoding categories as numbers.** Machine learning models don't understand text labels like "Velachery" or "SINGLE" and they need numbers. We'll convert each categorical column into a numeric representation, using the right method for each (ordinal numbers for room type, a statistically sound target encoding for locality, one hot encoding for gender and parking).

- **Handling skewed values.** Rent and deposit both have a long right tail, and a small number of very expensive listings stretches the scale. A logarithmic transformation compresses this tail and makes the model's job easier.

- **Splitting the data** into training, validation, and test sets. The model is then evaluated on data it has never seen, giving an honest picture of how it will perform in the real world.

After preprocessing, baseline models will be trained and the feature importance rankings that EDA estimated will be confirmed with actual model output.

---

## 8. Glossary

**Bivariate analysis**
Studying the relationship between exactly two things at a time. For example, how rent changes as occupancy type changes.

**Boolean (Yes or No) column**
A column that can only hold one of two values: True or False (or equivalently, Yes or No, 1 or 0). In this dataset, most amenity columns are boolean.

**Correlation**
A number between negative 1 and positive 1 that measures how strongly two things move together. Positive 1 means they move in perfect lockstep, 0 means no relationship, and negative 1 means when one goes up the other goes down. In this dataset, deposit and rent have a correlation of 0.44.

**Data error / outlier**
An entry in the dataset that doesn't match reality: a typo, a misplaced value, or a system glitch. The ₹2,00,000 deposit on a bare bones four sharing room is a data error. A ₹65,000 rent on a luxury single room is a real outlier, unusual but genuine.

**Distribution**
The pattern of how values spread out. A "right skewed distribution" means most values are low, but a few very high values pull the average upward. Rent in this dataset has a right skewed distribution.

**EDA (Exploratory Data Analysis)**
The process of studying a dataset before building any model, finding gaps, errors, patterns, and relationships. Think of it as thoroughly reading a document before using it.

**Encoding**
Converting non numeric data (like text categories) into numbers so a machine learning model can process them. For example, converting "SINGLE", "DOUBLE", "THREE", "FOUR" into 1, 2, 3, 4.

**Feature**
Any column in the dataset used as an input to the model. Rent is not a feature. It's the target. Everything else (locality, occupancy, amenities, etc.) is a feature.

**Imputation**
Filling in a missing value with a reasonable estimate rather than dropping the entire row. In this project, transit scores were imputed using the median score within the same locality.

**Interaction effect**
When the impact of one feature on price depends on the value of another feature. In this project, room type has a strong effect on rent in Sholinganallur but almost no effect in New Perungalathur. That is an interaction between locality and occupancy.

**log1p transformation**
A mathematical operation that compresses a wide, skewed range of numbers into a tighter range, making patterns easier for a model to learn. Applied to rent and deposit in preprocessing.

**Median**
The middle value when all entries are sorted from lowest to highest. More reliable than the average when a few extreme values would otherwise pull the average in a misleading direction.

**Multicollinearity**
When two features are strongly correlated with each other, making it hard for a model to tell which one is actually driving the outcome. Transit score and latitude are moderately collinear in this dataset (0.52 correlation).

**Multivariate analysis**
Studying how multiple features behave together simultaneously, going beyond any single pairwise comparison.

**NoBroker**
An Indian property listing platform that connects tenants directly with property owners, without a broker intermediary. The source of the data used in this project.

**Occupancy**
How many people share a room. SINGLE = just you. DOUBLE = two people. THREE = three people. FOUR = four people sharing the same room.

**Ordinal encoding**
A type of encoding where the numbers assigned to categories reflect a meaningful order. For example, SINGLE=1, DOUBLE=2, THREE=3, FOUR=4, where the numbers represent the sharing count, not arbitrary labels.

**Overfitting**
When a model learns the training data too well, including its noise and quirks, and then performs poorly on new data it hasn't seen. A particular risk with thin localities that have only a handful of listings.

**PG (Paying Guest)**
A rented room, often shared, where the tenant pays a monthly fee that typically includes basic amenities. Common across Indian cities, especially popular with students and young working professionals.

**Sentinel value**
A special number used to signal something other than its face value. In this dataset, NoBroker used negative 10 in the transit score column to mean "no data available", not an actual score of negative ten.

**Shrinkage / Smoothed target encoding**
A technique for encoding categorical columns with many categories (like 36 localities). Instead of using the raw average rent per locality (which would be unreliable for localities with only a few listings), the encoding blends the locality's own average with the overall dataset average. Localities with few listings get pulled closer to the overall mean; localities with many listings are trusted more.

**Skewness**
A measure of how asymmetric a distribution is. A skewness of 0 means perfectly symmetric. Rent in this dataset has a skewness of 4.92, indicating a strong right skew, with most listings cheap and a few very expensive ones.

**Target variable**
The thing the model is trying to predict. In this project, the target is monthly rent.

**Univariate analysis**
Studying one feature at a time: what does its distribution look like, are there outliers, does it make sense on its own?